# MapBiomas Argentina — Fuego Colección 1
## Paso 03 · métricas de la serie temporal de probabilidad de quema — exportación distribuida

### 📒 Si nunca usaste un notebook, leé esto primero

Este archivo es un **notebook** (corre en Google Colab) y mezcla dos tipos de contenido:

- **Texto**, como esta sección — son las instrucciones, no se ejecutan.
- **Celdas de código**, los bloques grises de más abajo. Cada una arranca con un comentario que la identifica (`# Celda 1 …`, `# Celda 2 …`, etc.).

Para **ejecutar** una celda, pasá el mouse por encima y tocá el **botón ▶ (play)** que aparece arriba a la izquierda (o seleccionala y apretá `Shift + Enter`).

Reglas básicas:

- **Ejecutá las celdas en orden y de a una.** Esperá a que **termine** una antes de correr la siguiente.
- Mientras una celda corre, el botón ▶ se convierte en un **círculo girando**. Cuando termina, aparece un **número entre corchetes** (`[1]`, `[2]`, …) a la izquierda y, si todo salió bien, no muestra errores en rojo.
- Al terminar, **cada celda imprime un mensaje informativo justo debajo** (por ejemplo *"repositorio clonado, funciones importadas — listo"* o *"N tarea(s) enviada(s)"*). Leelo: te confirma que salió bien y, a veces, qué hacer después.
- La **primera vez** que abrís Colab puede pedirte permiso para ejecutar / conectarte: aceptá.
- **Cuando termines, salí sin guardar los cambios.** No hace falta conservar nada de lo que editaste (por ejemplo el año en la Celda 3); si Colab te ofrece guardar, decí que no.

> Más abajo te decimos exactamente **qué celdas correr y en qué orden** según lo que quieras hacer (lanzar la exportación o solo mirar el progreso).

### ① Antes de correr algo, verificá estas tres cosas

1. **Agregá el proyecto de GEE (una sola vez).** En el [Code Editor](https://code.earthengine.google.com/), andá a la pestaña **Assets** → **ADD A PROJECT** → en el campo **Project** escribí `mapbiomas-argentina` y debería aparecer **`mapbiomas-argentina`**. Seleccionalo y tocá **Refresh**. Hacé esto **antes** de lanzar las tareas.
2. **¿Hay años disponibles?** Mirá la [tabla de exports](https://docs.google.com/spreadsheets/d/1-WRaIaKnmrVQktfxTngELpGshdx_AgNmY70kTH2XH3Q/edit?gid=0#gid=0) (planilla *GEE exports*, primera hoja). Solo podés tomar un año cuyo **Estado** sea **`Disponible`**. Los estados posibles:
   - **`Disponible`** — listo para ejecutar (lo podés tomar)
   - **`Corriendo`** — alguien ya lo está corriendo (no lo toques)
   - **`Completo`** — ya terminó sin problemas
   - **`Incompleto`** — corrió pero faltaron tiles o hubo errores (se puede retomar)
   - **`No habilitado`** — todavía no se puede correr este año
3. **¿Estás habilitado?** Tu **nombre y correo** tienen que estar en la [tabla de permisos](https://docs.google.com/spreadsheets/d/1_kphhLSGw2J8bSSV38NqSb1aeZrQs4z8FAIZ1fzuosw/edit?gid=0#gid=0). Sin eso la exportación falla de inmediato con un error de permisos. Si no estás, pedilo por el grupo de WhatsApp.

### ② Si se cumplen las tres, reclamá un año y ponelo a correr

1. En la [tabla de exports](https://docs.google.com/spreadsheets/d/1-WRaIaKnmrVQktfxTngELpGshdx_AgNmY70kTH2XH3Q/edit?gid=0#gid=0): escribí **tu nombre** en la fila del año y cambiá su **Estado** de `Disponible` a **`Corriendo`**.
2. Poné ese año en la **Celda 3** (más abajo) y corré las celdas en orden: **1 → 2 → 3 → 4**. Iniciá sesión con tu cuenta de Google cuando lo pida.
3. Cuando la **Celda 4** diga *"N tarea(s) enviada(s)"*, ya está: las tareas corren en los servidores de Google y **podés cerrar la pestaña, o incluso apagar tu PC** (salí sin guardar los cambios).
4. Al terminar (ver punto ③), volvé a la tabla y poné el Estado en **`Completo`** (o `Incompleto` si faltaron tiles / hubo errores; anotá el detalle en *Observaciones*).

> ⏳ **Esto suele tardar ~3.5 días.** GEE corre solo ~3 tareas a la vez por cuenta y un año son ~248 tiles, así que no esperes que termine en el momento.

### ③ Cómo ver el progreso más tarde (importante)

El entorno de Colab **caduca** tras un rato de inactividad o unas horas. Cuando volvés, la sesión está reseteada y se perdieron las variables — **pero las exportaciones NO se cancelan, siguen corriendo en los servidores de GEE**. Para chequear el estado más tarde:

1. Volvé a abrir este notebook y corré la **Celda 1** (reinstala e importa el código).
2. Corré la **Celda 2** — **tenés que volver a autenticarte** (la sesión caducó).
3. Volvé a poner tu año en la **Celda 3** y correla.
4. Corré la **Celda 5** (progreso). **NO corras la Celda 4**: esa es la que *envía* las tareas, y no hace falta para solo mirar el estado.
   - Excepción: si la Celda 5 dice que faltan tiles y querés **retomarlos**, ahí sí corré la **Celda 4** — solo reenvía lo que falta, no duplica lo ya exportado.

In [ ]:
# Celda 1 — Configuración. Ejecutar una vez por sesión (también al volver después de que Colab caduque).
!pip install -q -U earthengine-api
!git clone --depth 1 -b main https://github.com/barberaivan/mapbiomas-argentina-fire.git 2>/dev/null || (cd mapbiomas-argentina-fire && git pull -q)

import sys
sys.path.insert(0, '/content/mapbiomas-argentina-fire/collection-01')
from utils import functions as F
print('repositorio clonado, funciones importadas — listo')

In [ ]:
# Celda 2 — Autenticación. Iniciá sesión con tu cuenta de Google (la que figura en la tabla de permisos).
# Al volver después de que Colab caduque hay que autenticarse de nuevo.
# NOTA: Google te va a pedir permisos amplios, incluido Drive ("Ver, modificar y eliminar archivos de
# Google Drive"). Es parte del login estándar de Earth Engine; aceptá. Este notebook solo escribe en
# assets de GEE, NO toca tu Drive.
GEE_PROJECT = 'mapbiomas-argentina'

import ee
ee.Authenticate()
ee.Initialize(project=GEE_PROJECT)
print('inicializado — proyecto de cómputo:', GEE_PROJECT)

In [ ]:
# Celda 3 — ✏️ Poné acá el año que reclamaste en la tabla.
YEAR = 2003

In [ ]:
# Celda 4 — Enviar la exportación. (Para SOLO chequear el estado más tarde NO hace falta correr esta celda.)
tasks = F.bpts(year=YEAR)          # envía todos los tiles del AÑO; omite los ya exportados
print(len(tasks), 'tarea(s) enviada(s) para', YEAR, '— corren en los servidores de GEE; podés cerrar esta pestaña.')

In [ ]:
# Celda 5 — Ver el progreso. Corréla cuando quieras (después de las celdas 1, 2 y 3).
# Cuando diga que está completo, marcá el año como 'Completo' en la tabla de exports.
F.bpts_status(YEAR)

## Notas y resolución de problemas

- **Monitoreo:** las tareas enviadas también aparecen en la pestaña **Tasks** del [Code Editor](https://code.earthengine.google.com/) (misma cuenta de Google). `F.bpts_status(YEAR)` es la verificación más rápida — cuenta los assets terminados y lista lo que falta.
- **Ritmo:** GEE corre solo ~3 tareas a la vez por cuenta, y un año son ~248 tiles, así que un año reclamado tarda alrededor de 3,3 días en completarse. Ese es justamente el motivo de repartir el trabajo por persona/año.
- **Retomar un año:** si algunos tiles fallaron, simplemente volvé a ejecutar la celda `F.bpts(year=YEAR)` — solo se reenvían los tiles que faltan.
- **Cerrar la pestaña está bien** una vez enviadas las tareas; el trabajo continúa en el servidor.
- **Permisos (la causa más común de error):** tu nombre y correo tienen que estar en la [tabla de permisos](https://docs.google.com/spreadsheets/d/1_kphhLSGw2J8bSSV38NqSb1aeZrQs4z8FAIZ1fzuosw/edit?gid=0#gid=0). Tu cuenta de Google tiene que poder *leer* los assets de entrada y *escribir* en la colección de salida `projects/mapbiomas-argentina/assets/FIRE/COLLECTION-1/WORKFLOW-EXPORTS/bp_ts_metrics`. Si una ejecución falla de inmediato con un error de permisos, eso es lo que falta — pedí por el grupo de WhatsApp que te den acceso de escritura.